# Task 4.2 — Repository & Reproducibility

**Team:** Luís Serrano (60253), Tiago Fonseca (72898), Miguel Teixeira (72922)

This notebook handles all Task 4.2 deliverables:
- `environment.yml` — reproducible Conda environment specification
- `run_all.py` — end-to-end execution script for all project notebooks
- `experiments.csv` — final verification and tidy-up of the experiment log
- Submission checklist

**Run order:** execute this notebook **last**, after all other notebooks.

---

## 0. Setup

In [1]:
import sys
import numpy as np
import pandas as pd
import sklearn
import matplotlib
from pathlib import Path

# ── Path detection ─────────────────────────────────────────────────────────
_cwd = Path('.').resolve()
_tables_dir = None
for _p in [_cwd] + list(_cwd.parents):
    if (_p / 'tables' / 'experiments.csv').exists():
        _tables_dir = _p / 'tables'; break
    if (_p / 'experiments.csv').exists():
        _tables_dir = _p; break
TABLES_DIR = _tables_dir or _cwd
PROJECT_ROOT = TABLES_DIR.parent
SRC_DIR      = PROJECT_ROOT / 'src'
FIGURES_DIR  = PROJECT_ROOT / 'figures'

print(f'Project root : {PROJECT_ROOT.resolve()}')
print(f'Tables dir   : {TABLES_DIR.resolve()}')
print(f'Figures dir  : {FIGURES_DIR.resolve()}')
print()
print('Package versions:')
print(f'  Python       {sys.version.split()[0]}')
print(f'  NumPy        {np.__version__}')
print(f'  Pandas       {pd.__version__}')
print(f'  Scikit-learn {sklearn.__version__}')
print(f'  Matplotlib   {matplotlib.__version__}')

Project root : C:\Users\tfons\OneDrive\Ambiente de Trabalho\usl_project1_72922_72898_60253
Tables dir   : C:\Users\tfons\OneDrive\Ambiente de Trabalho\usl_project1_72922_72898_60253\tables
Figures dir  : C:\Users\tfons\OneDrive\Ambiente de Trabalho\usl_project1_72922_72898_60253\figures

Package versions:
  Python       3.11.9
  NumPy        2.3.5
  Pandas       2.2.2
  Scikit-learn 1.7.2
  Matplotlib   3.10.7


## 1. `environment.yml`

In [2]:
np_minor  = np.__version__.rsplit('.', 1)[0]
pd_minor  = pd.__version__.rsplit('.', 1)[0]
sk_minor  = sklearn.__version__.rsplit('.', 1)[0]
mpl_minor = matplotlib.__version__.rsplit('.', 1)[0]

env_lines = [
    'name: usl_project',
    'channels:',
    '  - conda-forge',
    '  - defaults',
    'dependencies:',
    '  - python=3.10',
    f'  - numpy>={np_minor}',
    f'  - pandas>={pd_minor}',
    f'  - scikit-learn>={sk_minor}',
    f'  - matplotlib>={mpl_minor}',
    '  - seaborn>=0.12',
    '  - scipy>=1.10',
    '  - jupyterlab>=4.0',
    '  - nbconvert',
    '  - ipykernel',
]

env_content = '\n'.join(env_lines) + '\n'
env_path = PROJECT_ROOT / 'environment.yml'
env_path.write_text(env_content, encoding='utf-8')
print(f'Saved: {env_path.resolve()}')
print()
print(env_content)

Saved: C:\Users\tfons\OneDrive\Ambiente de Trabalho\usl_project1_72922_72898_60253\environment.yml

name: usl_project
channels:
  - conda-forge
  - defaults
dependencies:
  - python=3.10
  - numpy>=2.3
  - pandas>=2.2
  - scikit-learn>=1.7
  - matplotlib>=3.10
  - seaborn>=0.12
  - scipy>=1.10
  - jupyterlab>=4.0
  - nbconvert
  - ipykernel



## 2. `run_all.py`

Executes all notebooks in the correct order to regenerate results end-to-end.

In [3]:
run_all_lines = [
    '#!/usr/bin/env python',
    '# run_all.py -- Regenerate all project results end-to-end.',
    '#',
    '# Usage:',
    '#   python run_all.py              -- full run (all tasks + extensions)',
    '#   python run_all.py --core-only  -- core tasks only',
    '#',
    '# Execution order:',
    '#   1. src/project.ipynb                  -- Task 1: K-Means baseline',
    '#   2. src/task2_gmm.ipynb                -- Task 2 + 3.2: GMM + AIC/BIC',
    '#   3. src/task3_evaluation.ipynb         -- Task 3.1 + 3.3 + 3.4',
    '#   4. src/task4_repository.ipynb         -- Task 4.2: repo deliverables',
    '#   5. src/extension_E2_spectral.ipynb    -- Extension E2: Spectral clustering',
    '#   6. src/extension_E5_visualization.ipynb -- Extension E5: t-SNE',
    '',
    'import argparse, subprocess, sys',
    'from pathlib import Path',
    '',
    'SRC_DIR = Path(__file__).parent / "src"',
    '',
    'CORE_NOTEBOOKS = [',
    '    SRC_DIR / "project.ipynb",',
    '    SRC_DIR / "task2_gmm.ipynb",',
    '    SRC_DIR / "task3_evaluation.ipynb",',
    '    SRC_DIR / "task4_repository.ipynb",',
    ']',
    '',
    'EXTENSION_NOTEBOOKS = [',
    '    SRC_DIR / "extension_E2_spectral.ipynb",',
    '    SRC_DIR / "extension_E5_visualization.ipynb",',
    ']',
    '',
    '',
    'def run_notebook(nb):',
    '    print(f"\\n" + "="*60 + f"\\nRunning: {nb.name}\\n" + "="*60)',
    '    result = subprocess.run(',
    '        [sys.executable, "-m", "jupyter", "nbconvert",',
    '         "--to", "notebook", "--execute", "--inplace",',
    '         "--ExecutePreprocessor.timeout=7200", str(nb)],',
    '        cwd=str(nb.parent),',
    '    )',
    '    if result.returncode != 0:',
    '        print(f"ERROR: {nb.name} failed (exit {result.returncode})")',
    '        sys.exit(result.returncode)',
    '    print(f"Done: {nb.name}")',
    '',
    '',
    'if __name__ == "__main__":',
    '    parser = argparse.ArgumentParser()',
    '    parser.add_argument("--core-only", action="store_true")',
    '    args = parser.parse_args()',
    '    notebooks = CORE_NOTEBOOKS if args.core_only else CORE_NOTEBOOKS + EXTENSION_NOTEBOOKS',
    '    print(f"Running {len(notebooks)} notebook(s)")',
    '    for nb in notebooks:',
    '        if nb.exists():',
    '            run_notebook(nb)',
    '        else:',
    '            print(f"WARNING: {nb} not found")',
    '    print("\\n Done. All results in tables/ and figures/.")',
]

run_all_content = '\n'.join(run_all_lines) + '\n'
run_all_path = PROJECT_ROOT / 'run_all.py'
run_all_path.write_text(run_all_content, encoding='utf-8')
print(f'Saved: {run_all_path.resolve()}')
print(f'Lines: {len(run_all_lines)}')

Saved: C:\Users\tfons\OneDrive\Ambiente de Trabalho\usl_project1_72922_72898_60253\run_all.py
Lines: 59


## 3. Experiment log verification

In [4]:
exp_path = TABLES_DIR / 'experiments.csv'
exp_df   = pd.read_csv(exp_path)

print(f'experiments.csv: {len(exp_df)} rows')
print()

method_summary = (exp_df.groupby('method')
                  .agg(n_runs=('run_id','count'),
                       representations=('representation_id',
                                        lambda s: ', '.join(s.unique())))
                  .reset_index())
print('Runs by method:')
display(method_summary)

print('\nAll experiment IDs:')
print(exp_df[['run_id','method','parameters']].to_string(index=False))

experiments.csv: 53 rows

Runs by method:


,method,n_runs,representations
0,gmm,9,final_precluster_matrix
1,ikmeans,1,final_precluster_matrix
2,kmeans,13,"final_precluster_matrix, robust_final_preclust..."
3,spectral_tutorial,24,final_precluster_matrix_dense3000
4,tsne,6,final_precluster_matrix



All experiment IDs:
 run_id            method                                                                                                                                                                              parameters
exp_001            kmeans                                                                                                           n_clusters=3; init=not_explicit_in_notebook; n_init=20; max_iter=300; runs=10
exp_002            kmeans                                                                                                           n_clusters=4; init=not_explicit_in_notebook; n_init=20; max_iter=300; runs=10
exp_003            kmeans                                                                                                           n_clusters=5; init=not_explicit_in_notebook; n_init=20; max_iter=300; runs=10
exp_004            kmeans                                                                                                           n_clust

## 4. File structure verification

In [5]:
print('=== DELIVERABLE FILE CHECK ===')

notebooks_expected = [
    'project.ipynb', 'task2_gmm.ipynb', 'task3_evaluation.ipynb',
    'task4_repository.ipynb', 'extension_E2_spectral.ipynb',
    'extension_E5_visualization.ipynb',
]
print('\nNotebooks (src/):')
for nb in notebooks_expected:
    p = SRC_DIR / nb
    print(f'  {"OK" if p.exists() else "MISSING":6s}  {nb}')

tables_expected = [
    # Core / representation
    'experiments.csv',
    'final_precluster_matrix.csv',
    'feature_selection_table.csv',
    'feature_treatment_table.csv',
    'missingness_table.csv',
    'outlier_table.csv',
    'pipeline_sketch.csv',
    'distance_contribution_by_family.csv',   # created by project.ipynb (re-run needed)
    # K-Means / iKMeans baselines
    'baseline_results.csv',
    'baseline_run_results.csv',
    'ikmeans_cluster_share.csv',
    'cluster_summary.csv',
    'cluster_numeric_profile.csv',
    'cluster_categorical_profile.csv',
    'cluster_posthoc_profile.csv',
    # Representation sensitivity
    'representation_sensitivity_summary.csv',
    'representation_sensitivity_ari.csv',
    'representation_sensitivity_ari_summary.csv',
    # GMM (Task 2 + 3.2)
    'gmm_aic_bic.csv',
    'gmm_labels.csv',
    'gmm_seed_stability.csv',
    'gmm_stability_summary.csv',
    'gmm_repr_sensitivity.csv',
    'gmm_cluster_summary.csv',
    # Task 3 evaluation
    'cross_method_ari.csv',
    'internal_validity_comparison.csv',
    # Extension E2 – Spectral
    'spectral_summary.csv',
    'spectral_sweep.csv',
    'spectral_seed_stability.csv',
    'spectral_cross_method_ari.csv',
    'spectral_kgraph_robustness.csv',
    'spectral_validity_comparison.csv',
    # Extension E5 – t-SNE
    'tsne_trustworthiness.csv',
]
print('\nTables:')
for t in tables_expected:
    p = TABLES_DIR / t
    print(f'  {"OK" if p.exists() else "MISSING":6s}  {t}')

figures_expected = [
    # Task 2/3 – GMM
    'gmm_aic_bic.png',
    'internal_validity_comparison.png',
    'gmm_seed_stability.png',
    'cross_method_ari_heatmap.png',
    'cluster_share_comparison.png',
    'gmm_cluster_profile_heatmap.png',
    'gmm_membership_distribution.png',
    # Extension E2 – Spectral
    'spectral_eigengap.png',
    'spectral_embedding_2d.png',
    'spectral_validity_comparison.png',
    'spectral_cross_method_heatmap.png',
    'spectral_kgraph_robustness.png',
    # Extension E5 – t-SNE
    'tsne_kmeans_clusters.png',
    'tsne_feature_overlay.png',
    'tsne_sensitivity.png',
]
print('\nFigures:')
for f in figures_expected:
    p = FIGURES_DIR / f
    print(f'  {"OK" if p.exists() else "MISSING":6s}  {f}')

print('\nRoot files:')
for r in ['environment.yml', 'run_all.py']:
    p = PROJECT_ROOT / r
    print(f'  {"OK" if p.exists() else "MISSING":6s}  {r}')

missing_tables  = [t for t in tables_expected  if not (TABLES_DIR / t).exists()]
missing_figures = [f for f in figures_expected if not (FIGURES_DIR / f).exists()]
print(f'\nSummary: {len(missing_tables)} missing table(s), {len(missing_figures)} missing figure(s)')
if missing_tables:
    print('  Missing tables :', missing_tables)
if missing_figures:
    print('  Missing figures:', missing_figures)

=== DELIVERABLE FILE CHECK ===

Notebooks (src/):
  OK      project.ipynb
  OK      task2_gmm.ipynb
  OK      task3_evaluation.ipynb
  OK      task4_repository.ipynb
  OK      extension_E2_spectral.ipynb
  OK      extension_E5_visualization.ipynb

Tables:
  OK      experiments.csv
  OK      final_precluster_matrix.csv
  OK      feature_selection_table.csv
  OK      feature_treatment_table.csv
  OK      missingness_table.csv
  OK      outlier_table.csv
  OK      pipeline_sketch.csv
  MISSING  distance_contribution_by_family.csv
  OK      baseline_results.csv
  OK      baseline_run_results.csv
  OK      ikmeans_cluster_share.csv
  OK      cluster_summary.csv
  OK      cluster_numeric_profile.csv
  OK      cluster_categorical_profile.csv
  OK      cluster_posthoc_profile.csv
  OK      representation_sensitivity_summary.csv
  OK      representation_sensitivity_ari.csv
  OK      representation_sensitivity_ari_summary.csv
  OK      gmm_aic_bic.csv
  OK      gmm_labels.csv
  OK      gmm_seed_

## 5. Submission checklist

| # | Task | Notebook | Status |
|---|---|---|---|
| 1 | K-Means baseline (k=3..8, iKMeans) | `project.ipynb` | ✅ |
| 1+ | Dataset fingerprint (MD5 + SHA-256) | `project.ipynb` | ✅ |
| 1+ | Distance contribution by feature family | `project.ipynb` | ✅ |
| 1+ | Feature selection table with encoding column | `project.ipynb` | ✅ |
| 1+ | Formal representation ID (`R-EUCLID-standard-countryGrouped-noADR`) | `project.ipynb` | ✅ |
| 2 | GMM (diag covariance, BIC selection) | `task2_gmm.ipynb` | ✅ |
| 3.1 | Internal validity comparison (Sil / CH / DB) | `task3_evaluation.ipynb` | ✅ |
| 3.2 | AIC / BIC diagnostic curves | `task2_gmm.ipynb` | ✅ |
| 3.3 | Stability (5 seeds, RobustScaler, cross-ARI) | `task3_evaluation.ipynb` | ✅ |
| 3.4 | Cluster profiles + soft membership | `task3_evaluation.ipynb` | ✅ |
| 4.2 | `environment.yml`, `run_all.py`, `experiments.csv` | `task4_repository.ipynb` | ✅ |
| E2 | Spectral (tutorial pipeline, kNN affinity, dense eigh) | `extension_E2_spectral.ipynb` | ✅ |
| E5 | t-SNE nonlinear visualisation | `extension_E5_visualization.ipynb` | ✅ |
| — | Academic report (6–8 pages) | — | ⬜ TODO |
| — | `git tag Submission` | — | ⬜ TODO |

In [6]:
print('=' * 60)
print('  TASK 4.2 COMPLETE')
print('=' * 60)
print(f'  environment.yml : {(PROJECT_ROOT / "environment.yml").exists()}')
print(f'  run_all.py      : {(PROJECT_ROOT / "run_all.py").exists()}')
print(f'  experiments.csv : {len(exp_df)} runs across {exp_df["method"].nunique()} methods')
print(f'                    methods: {sorted(exp_df["method"].unique())}')
print()

missing = [t for t in tables_expected if not (TABLES_DIR / t).exists()]
if missing:
    print(f'  WARNING: {len(missing)} table(s) not yet generated:')
    for m in missing:
        print(f'    - {m}')
    print('  → Re-run project.ipynb to generate them.')
else:
    print('  All expected tables: OK')

missing_fig = [f for f in figures_expected if not (FIGURES_DIR / f).exists()]
if missing_fig:
    print(f'  WARNING: {len(missing_fig)} figure(s) missing: {missing_fig}')
else:
    print('  All expected figures: OK')

print()
print('  Remaining TODO:')
print('    [ ] Write academic report (6–8 pages)')
print('    [ ] git add . && git commit -m "Final submission"')
print('    [ ] git tag Submission')
print('=' * 60)

  TASK 4.2 COMPLETE
  environment.yml : True
  run_all.py      : True
  experiments.csv : 53 runs across 5 methods
                    methods: ['gmm', 'ikmeans', 'kmeans', 'spectral_tutorial', 'tsne']

    - distance_contribution_by_family.csv
  → Re-run project.ipynb to generate them.
  All expected figures: OK

  Remaining TODO:
    [ ] Write academic report (6–8 pages)
    [ ] git add . && git commit -m "Final submission"
    [ ] git tag Submission
